In [1]:
import sys
import os

if 'google.colab' in sys.modules:
    print("☁️ Google Colab detected. Setting up GitHub repository...")
    if not os.path.exists('manero-panganiban-serafica'):
        !git clone https://github.com/kricme/manero-panganiban-serafica.git
    else:
        print("🔄 Repository already exists. Pulling latest updates from GitHub...")
        !git -C manero-panganiban-serafica pull

    if not os.getcwd().endswith('05 Speech and Econ Dataset Merging'):
        %cd "manero-panganiban-serafica/05 Speech and Econ Dataset Merging"
else:
    print("💻 Local environment detected. Using local file paths.")

☁️ Google Colab detected. Setting up GitHub repository...
🔄 Repository already exists. Pulling latest updates from GitHub...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 12 (delta 9), reused 12 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 414.41 KiB | 1.04 MiB/s, done.
From https://github.com/kricme/manero-panganiban-serafica
   addf7f1..f1b0bf6  main       -> origin/main
Updating addf7f1..f1b0bf6
Fast-forward
 .../withGDP_Econ_Dataset_with_orig_and_yoy.csv     |  582 ++--
 .../withoutGDP_Econ_Dataset_with_orig_and_yoy.csv  |  636 ++--
 .../03 economic dataset/withGDP_Econ_Dataset.csv   |  582 ++--
 .../withoutGDP_Econ_Dataset.csv                    |  636 ++--
 ...e-Processing, Merging, and Transformation.ipynb | 3578 ++++++++++----------
 .../03 EDA for Speech and Econ Datasets.ipynb      |  794 ++---
 6 files changed, 3392 insertions(+), 3416 deletions(-)
/conte

In [2]:
import pandas as pd

In [3]:
# Load Datasets

rate_df = pd.read_csv("../00 Cleaned Dataset/03 economic dataset/withGDP_Econ_Dataset.csv")
sentiment_df = pd.read_csv("../00 Cleaned Dataset/02 fine-tuned bert results/econobert_speeches.csv")

In [4]:
sentiment_df.info()
sentiment_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 673 entries, 0 to 672
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           673 non-null    int64  
 1   Date            673 non-null    object 
 2   tone_mean       673 non-null    float64
 3   tone_median     673 non-null    float64
 4   pos_share       673 non-null    float64
 5   neg_share       673 non-null    float64
 6   neu_share       673 non-null    float64
 7   n_sent          673 non-null    int64  
 8   label_majority  673 non-null    object 
dtypes: float64(5), int64(2), object(2)
memory usage: 47.4+ KB


,index,Date,tone_mean,tone_median,pos_share,neg_share,neu_share,n_sent,label_majority
0,0,2020-03-03,0.553776,0.920367,0.670103,0.103093,0.216495,97,positive
1,1,2020-02-28,0.553565,0.836266,0.592000,0.048000,0.360000,125,positive
2,2,2020-02-27,0.603716,0.935497,0.718750,0.078125,0.187500,128,positive
3,3,2020-02-06,0.494486,0.871817,0.590909,0.090909,0.284091,88,positive
4,4,2020-01-31,0.135191,0.058430,0.100000,0.000000,0.900000,20,neutral


In [5]:
#dropping unnecessary columns for the succeeding analysis
sentiment_df = sentiment_df.drop(columns=[
    'tone_median',
    'n_sent',
    'label_majority'
    ])
sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 673 entries, 0 to 672
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      673 non-null    int64  
 1   Date       673 non-null    object 
 2   tone_mean  673 non-null    float64
 3   pos_share  673 non-null    float64
 4   neg_share  673 non-null    float64
 5   neu_share  673 non-null    float64
dtypes: float64(4), int64(1), object(1)
memory usage: 31.7+ KB


In [6]:
rate_df.info()
rate_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      288 non-null    object 
 1   Interbank Call Loan Rate  288 non-null    float64
 2   Real GDP                  288 non-null    float64
 3   CPI                       288 non-null    float64
 4   Wholesale Price           288 non-null    float64
 5   Industrial Production     288 non-null    float64
 6   Intl Trade Merch Exports  288 non-null    float64
 7   Intl Trade Merch Imports  288 non-null    float64
 8   FX Rate                   288 non-null    float64
dtypes: float64(8), object(1)
memory usage: 20.4+ KB


,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate
0,2001-04-01,10.585227,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502
1,2001-05-01,9.875000,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137
2,2001-06-01,9.343750,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388
3,2001-07-01,9.092187,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097
4,2001-08-01,9.196023,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568


In [7]:
def to_month_start(s: pd.Series) -> pd.Series:
    """Convert date-like series to month-start timestamps (monthly frequency)."""
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp()

rate = rate_df.copy()
rate["Date"] = pd.to_datetime(rate["Date"], errors="coerce")
rate = rate.dropna(subset=["Date"])
rate["Date"] = to_month_start(rate["Date"])

In [8]:
date_counts = rate['Date'].value_counts()
if (date_counts > 1).any():
    print("Multiple rows exist for some months. Here are the months with more than one entry:")
    print(date_counts[date_counts > 1])
else:
    print("Each month has exactly one row in the `rate` DataFrame.")

Each month has exactly one row in the `rate` DataFrame.


In [9]:
# Ensure sorted (optional)
rate = rate.sort_values("Date").reset_index(drop=True)

In [10]:
# Aggregate sentiment_df to MONTHLY values
# - multiple speeches in month -> average value

sent = sentiment_df.loc[:, ["Date", "tone_mean", "pos_share", "neg_share", "neu_share"]].copy()
sent["Date"] = pd.to_datetime(sent["Date"], errors="coerce")
sent = sent.dropna(subset=["Date"])

sent_monthly = (
    sent.assign(Date=to_month_start(sent["Date"]))
        .groupby("Date", as_index=False)
        .agg(
            tone_mean=("tone_mean", "mean"),
            pos_mean=("pos_share", "mean"),
            neg_mean=("neg_share", "mean"),
            neu_mean=("neu_share", "mean"),
            speech_count=("tone_mean", "size")
        )
)

In [11]:
# Merge: keep MONTHLY macro data even if no speeches

merged = rate.merge(sent_monthly, on="Date", how="left").sort_values("Date").reset_index(drop=True)

In [12]:
merged

,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count
0,2001-04-01,10.585227,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502,0.407965,0.484390,0.069972,0.424209,5.0
1,2001-05-01,9.875000,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137,0.545883,0.646130,0.070870,0.271961,4.0
2,2001-06-01,9.343750,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388,0.434630,0.437514,0.009009,0.540656,3.0
3,2001-07-01,9.092187,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097,0.415988,0.487176,0.067964,0.441351,5.0
4,2001-08-01,9.196023,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568,0.395588,0.525983,0.123081,0.348079,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,2024-12-01,6.159405,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,NaN
284,2025-01-01,6.086828,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,NaN
285,2025-02-01,5.895856,5.379357,2.884608,2.911208,2.836625,14.402545,16.015169,3.619466,NaN,NaN,NaN,NaN,NaN
286,2025-03-01,5.923455,5.379357,2.071718,2.902758,-1.145631,16.907083,5.594593,2.820643,NaN,NaN,NaN,NaN,NaN


In [13]:
merged[merged['tone_mean'].isna()]

,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count
9,2002-01-01,8.851103,3.348574,4.527933,5.410123,-2.568323,-21.756533,7.624421,0.864068,NaN,NaN,NaN,NaN,NaN
10,2002-02-01,7.811080,3.876541,3.714286,5.484146,-8.365021,-7.668831,-30.090635,6.195906,NaN,NaN,NaN,NaN,NaN
14,2002-06-01,7.076707,3.660185,3.497161,5.625527,4.566641,10.817769,-1.057510,-2.100636,NaN,NaN,NaN,NaN,NaN
22,2003-02-01,7.067434,4.618477,2.754824,6.255075,14.834025,8.320999,56.766683,5.446697,NaN,NaN,NaN,NaN,NaN
23,2003-03-01,7.060855,4.618477,3.219869,6.467255,11.165525,11.927679,37.382946,6.902442,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,2024-12-01,6.159405,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,NaN
284,2025-01-01,6.086828,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,NaN
285,2025-02-01,5.895856,5.379357,2.884608,2.911208,2.836625,14.402545,16.015169,3.619466,NaN,NaN,NaN,NaN,NaN
286,2025-03-01,5.923455,5.379357,2.071718,2.902758,-1.145631,16.907083,5.594593,2.820643,NaN,NaN,NaN,NaN,NaN


In [14]:
# Fill months with no speeches by copying the previous available mean

merged["tone_mean_filled"] = merged["tone_mean"].ffill()
merged["pos_mean_filled"] = merged["pos_mean"].ffill()
merged["neg_mean_filled"] = merged["neg_mean"].ffill()
merged["neu_mean_filled"] = merged["neu_mean"].ffill()

# Make speech_count explicit (0 if no speeches that month)
merged["speech_count"] = merged["speech_count"].fillna(0).astype(int)

In [15]:
merged[merged['tone_mean'].isna()]

,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled
9,2002-01-01,8.851103,3.348574,4.527933,5.410123,-2.568323,-21.756533,7.624421,0.864068,NaN,NaN,NaN,NaN,0,0.297909,0.417388,0.145743,0.436869
10,2002-02-01,7.811080,3.876541,3.714286,5.484146,-8.365021,-7.668831,-30.090635,6.195906,NaN,NaN,NaN,NaN,0,0.297909,0.417388,0.145743,0.436869
14,2002-06-01,7.076707,3.660185,3.497161,5.625527,4.566641,10.817769,-1.057510,-2.100636,NaN,NaN,NaN,NaN,0,0.330122,0.421541,0.060662,0.507380
22,2003-02-01,7.067434,4.618477,2.754824,6.255075,14.834025,8.320999,56.766683,5.446697,NaN,NaN,NaN,NaN,0,0.151484,0.348433,0.214258,0.428135
23,2003-03-01,7.060855,4.618477,3.219869,6.467255,11.165525,11.927679,37.382946,6.902442,NaN,NaN,NaN,NaN,0,0.151484,0.348433,0.214258,0.428135
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283,2024-12-01,6.159405,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,0,0.553776,0.670103,0.103093,0.216495
284,2025-01-01,6.086828,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,0,0.553776,0.670103,0.103093,0.216495
285,2025-02-01,5.895856,5.379357,2.884608,2.911208,2.836625,14.402545,16.015169,3.619466,NaN,NaN,NaN,NaN,0,0.553776,0.670103,0.103093,0.216495
286,2025-03-01,5.923455,5.379357,2.071718,2.902758,-1.145631,16.907083,5.594593,2.820643,NaN,NaN,NaN,NaN,0,0.553776,0.670103,0.103093,0.216495


In [16]:
merged[merged['tone_mean_filled'].isna()]

,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled


In [17]:
print(f"Rate DataFrame Date Range: {rate['Date'].min().date()} to {rate['Date'].max().date()}")
print(f"Sentiment Monthly DataFrame Date Range: {sent_monthly['Date'].min().date()} to {sent_monthly['Date'].max().date()}")

Rate DataFrame Date Range: 2001-04-01 to 2025-04-01
Sentiment Monthly DataFrame Date Range: 1998-03-01 to 2020-03-01


In [18]:
start_date = '2001-04-01'
end_date = '2020-03-01'

# Filter the DataFrame
merged = merged[(merged['Date'] >= start_date) & (merged['Date'] <= end_date)]

print(f"Filtered Merged DataFrame Shape: {merged.shape}")
print(f"Filtered Merged DataFrame Date Range: {merged['Date'].min().date()} to {merged['Date'].max().date()}")

Filtered Merged DataFrame Shape: (228, 18)
Filtered Merged DataFrame Date Range: 2001-04-01 to 2020-03-01


In [19]:
merged.head()

,Date,Interbank Call Loan Rate,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled
0,2001-04-01,10.585227,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502,0.407965,0.484390,0.069972,0.424209,5,0.407965,0.484390,0.069972,0.424209
1,2001-05-01,9.875000,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137,0.545883,0.646130,0.070870,0.271961,4,0.545883,0.646130,0.070870,0.271961
2,2001-06-01,9.343750,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388,0.434630,0.437514,0.009009,0.540656,3,0.434630,0.437514,0.009009,0.540656
3,2001-07-01,9.092187,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097,0.415988,0.487176,0.067964,0.441351,5,0.415988,0.487176,0.067964,0.441351
4,2001-08-01,9.196023,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568,0.395588,0.525983,0.123081,0.348079,5,0.395588,0.525983,0.123081,0.348079


In [20]:
merged.to_csv('../00 Cleaned Dataset/04 econ data with econobert results/withGDP_IRD_with_tone_dataset.csv', index=False)